In [1]:
# !pip install gdown
# !pip install sacrebleu

In [2]:
# import gdown
import pandas as pd
import numpy as np
import torch
# import matplotlib.pyplot as plt
# import seaborn as sns
import unicodedata
import re

# import sacrebleu
import math

from datasets import load_dataset

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from torch.cuda import temperature

2026-03-23 15:42:55.527460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774280575.904826      31 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774280576.014743      31 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774280576.922791      31 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774280576.922840      31 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774280576.922844      31 computation_placer.cc:177] computation placer alr

In [3]:
# file_id = '1qdSb3fLeI8FsXoZbiKpxKCFA_fE-89sG'
# gdown.download(id=file_id, output='model.zip')

In [4]:
# model_path = "/kaggle/input/datasets/vitalykudelya/dpc26-models-2/LARGE_MIXES_WITH_OLD_NEW_DATA_v2_53.93/LARGE_MIXES_WITH_OLD_NEW_DATA_v2"
test_set_path = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
device = device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
scribal_characters = ["?", "!", ":", "/"]
parentheses_pattern = r'\(.*?\)'
angle_bracket_pattern = r'<.*?>'
square_bracket_pattern = r'\[.*?\]'
partial_broken_signs_pattern = r'\˹.*?\˺'
scribal_notation_patterns = [parentheses_pattern, angle_bracket_pattern, square_bracket_pattern, partial_broken_signs_pattern]
subscript_to_normal_map = {
    '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
    '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
    '⁰': '0', '¹': '1', '²': '2', '³': '3', '⁴': '4',
    '⁵': '5', '⁶': '6', '⁷': '7', '⁸': '8', '⁹': '9'
}
translit_character_map = {'Ţ': 'Ṭ', 'Ḫ': 'H', 'ḫ': 'h', 'ḥ':
                          'h', 'ṇ': 'n', 'ț': 'ṭ', 'ţ': 'ṭ',
                          'ş': 'ṣ', 'ș': 'ṣ', 'Ş': 'Ṣ',
                          'Ī': 'I', 'İ': 'I', 'Ü': 'Ù'}

In [6]:
"""
Ensemble MBR Pipeline
=====================
Этапы:
  1. Определение моделей (ModelConfig)
  2. Генерация кандидатов от каждой модели последовательно
  3. Сбор кандидатов в DataFrame
  4. MBR decoding — выбор лучшего кандидата
  5. (Опционально) Оценка на валидации
"""

import gc
import math
from dataclasses import dataclass, field
from typing import List, Optional, Dict

import pandas as pd
import numpy as np
import sacrebleu
import torch
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# ============================================================
# Этап 0: Конфигурация моделей
# ============================================================

@dataclass
class ModelConfig:
    """Описание одной модели для ensemble"""
    name: str                          # Человекочитаемое имя
    model_path: str                    # Путь к весам
    tokenizer_path: Optional[str] = None  # Путь к токенайзеру (если отличается)
    
    # Generation параметры
    num_beams: int = 12
    num_return: int = 5
    max_input_length: int = 600
    max_output_length: int = 690
    length_penalty: float = 1.7
    
    # Batch
    batch_size: int = 5
    
    # Доп. параметры загрузки
    torch_dtype: Optional[torch.dtype] = None  # например torch.bfloat16
    local_files_only: bool = True
    
    def __post_init__(self):
        if self.tokenizer_path is None:
            self.tokenizer_path = self.model_path


# ============================================================
# Этап 1: Генерация кандидатов от одной модели
# ============================================================

def generate_candidates_single_model(
    config: ModelConfig,
    texts: List[str],
    device: str = "cuda",
) -> List[List[str]]:
    """
    Загружает модель, генерирует кандидатов, освобождает память.
    
    Returns:
        List[List[str]]: для каждого текста — список из num_return кандидатов
    """
    print(f"\n{'='*60}")
    print(f"[{config.name}] Loading model from {config.model_path}")
    print(f"  beams={config.num_beams}, return={config.num_return}, "
          f"lp={config.length_penalty}, batch={config.batch_size}")
    print(f"{'='*60}")
    
    # Загрузка
    load_kwargs = {"local_files_only": config.local_files_only}
    if config.torch_dtype is not None:
        load_kwargs["torch_dtype"] = config.torch_dtype
    
    model = AutoModelForSeq2SeqLM.from_pretrained(config.model_path, **load_kwargs)
    model.to(device)
    model.eval()
    
    tokenizer = AutoTokenizer.from_pretrained(
        config.tokenizer_path, local_files_only=config.local_files_only
    )
    
    # Генерация
    all_candidates = []
    n_batches = math.ceil(len(texts) / config.batch_size)
    
    for i in tqdm(range(0, len(texts), config.batch_size),
                  total=n_batches, desc=f"[{config.name}] Generating"):
        batch_texts = texts[i : i + config.batch_size]
        
        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            max_length=config.max_input_length,
            padding=True,
            truncation=True,
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=config.num_beams,
                num_return_sequences=config.num_return,
                max_length=config.max_output_length,
                length_penalty=config.length_penalty,
                early_stopping=True,
            )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # Разбить по примерам: [batch_size * num_return] → batch_size × num_return
        for j in range(len(batch_texts)):
            start = j * config.num_return
            end = start + config.num_return
            cands = [d.strip() for d in decoded[start:end]]
            all_candidates.append(cands)
    
    # Освобождение памяти
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    
    mem_free = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()
    print(f"[{config.name}] Done. Generated {len(all_candidates)} × {config.num_return} candidates. "
          f"VRAM free: {mem_free / 1e9:.1f} GB")
    
    return all_candidates


# ============================================================
# Этап 2: Прогон всех моделей, сбор кандидатов в DataFrame
# ============================================================

def generate_all_candidates(
    model_configs: List[ModelConfig],
    texts: List[str],
    device: str = "cuda",
) -> pd.DataFrame:
    """
    Последовательно прогоняет все модели.
    
    Returns:
        DataFrame с колонками:
          - text_idx: индекс текста
          - input_text: исходный текст
          - {model_name}_0, {model_name}_1, ...: кандидаты от каждой модели
    """
    n = len(texts)
    
    # Базовый DataFrame
    df = pd.DataFrame({
        "text_idx": range(n),
        "input_text": texts,
    })
    
    # Прогон каждой модели
    for config in model_configs:
        candidates = generate_candidates_single_model(config, texts, device)
        
        # Добавить колонки: model_name_0, model_name_1, ...
        for k in range(config.num_return):
            col_name = f"{config.name}_{k}"
            df[col_name] = [candidates[i][k] if k < len(candidates[i]) else "" 
                           for i in range(n)]
    
    return df


def get_candidate_columns(df: pd.DataFrame) -> List[str]:
    """Вернуть имена колонок с кандидатами"""
    return [c for c in df.columns if c not in ("text_idx", "input_text")]


def get_candidates_for_row(row: pd.Series, candidate_cols: List[str]) -> List[str]:
    """Извлечь всех кандидатов для одного примера"""
    return [row[c] for c in candidate_cols if row[c] and str(row[c]).strip()]


# ============================================================
# Этап 3: MBR Decoding
# ============================================================

def mbr_decode_chrf(candidates: List[str]) -> str:
    """MBR по chrF++ (быстрее, стабильнее)"""
    if len(candidates) <= 1:
        return candidates[0] if candidates else ""
    
    # Дедупликация
    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    candidates = unique
    
    if len(candidates) == 1:
        return candidates[0]
    
    best_score = -1
    best_text = candidates[0]
    
    for i, hyp in enumerate(candidates):
        total = 0
        for j, ref in enumerate(candidates):
            if i != j:
                total += sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
        avg = total / (len(candidates) - 1)
        
        if avg > best_score:
            best_score = avg
            best_text = hyp
    
    return best_text


def mbr_decode_combined(candidates: List[str]) -> str:
    """MBR по geometric mean(BLEU, chrF++) — как метрика соревнования"""
    if len(candidates) <= 1:
        return candidates[0] if candidates else ""
    
    seen = set()
    unique = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            unique.append(c)
    candidates = unique
    
    if len(candidates) == 1:
        return candidates[0]
    
    best_score = -1
    best_text = candidates[0]
    
    for i, hyp in enumerate(candidates):
        chrf_total = 0
        bleu_total = 0
        
        for j, ref in enumerate(candidates):
            if i != j:
                chrf_total += sacrebleu.sentence_chrf(hyp, [ref], word_order=2).score
                bleu_total += sacrebleu.sentence_bleu(hyp, [ref], tokenize="intl").score
        
        n = len(candidates) - 1
        avg_chrf = chrf_total / n
        avg_bleu = bleu_total / n
        
        if avg_bleu > 0 and avg_chrf > 0:
            combined = math.sqrt(avg_bleu * avg_chrf)
        else:
            combined = avg_chrf
        
        if combined > best_score:
            best_score = combined
            best_text = hyp
    
    return best_text


# Доступные стратегии
MBR_STRATEGIES = {
    "chrf": mbr_decode_chrf,
    "combined": mbr_decode_combined,
}


def select_best_candidates(
    candidates_df: pd.DataFrame,
    strategy: str = "chrf",
) -> pd.DataFrame:
    """
    Применяет MBR decoding к DataFrame с кандидатами.
    
    Returns:
        DataFrame с колонками: text_idx, input_text, prediction
    """
    decode_fn = MBR_STRATEGIES[strategy]
    candidate_cols = get_candidate_columns(candidates_df)
    
    print(f"\nMBR Decoding (strategy={strategy})")
    print(f"  Candidates per example: {len(candidate_cols)}")
    print(f"  Total examples: {len(candidates_df)}")
    
    predictions = []
    for _, row in tqdm(candidates_df.iterrows(), 
                       total=len(candidates_df), desc="MBR Decoding"):
        cands = get_candidates_for_row(row, candidate_cols)
        best = decode_fn(cands)
        predictions.append(best)
    
    result = pd.DataFrame({
        "text_idx": candidates_df["text_idx"],
        "input_text": candidates_df["input_text"],
        "prediction": predictions,
    })
    
    return result


# ============================================================
# Этап 4: Оценка (валидация)
# ============================================================

def deep_past_score(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """Метрика соревнования: geometric mean(BLEU, chrF++)"""
    bleu = sacrebleu.corpus_bleu(predictions, [references], tokenize="intl").score
    chrf = sacrebleu.corpus_chrf(predictions, [references], word_order=2).score
    score = math.sqrt(bleu * chrf) if bleu > 0 and chrf > 0 else 0.0
    return {"bleu": bleu, "chrf++": chrf, "score": score}


def evaluate_predictions(
    predictions_df: pd.DataFrame,
    references: List[str],
) -> Dict[str, float]:
    """Оценить предсказания"""
    preds = predictions_df["prediction"].tolist()
    metrics = deep_past_score(preds, references)
    
    print(f"\n{'='*40}")
    print(f"BLEU:   {metrics['bleu']:.4f}")
    print(f"chrF++: {metrics['chrf++']:.4f}")
    print(f"Score:  {metrics['score']:.4f}")
    print(f"{'='*40}")
    
    return metrics


def evaluate_per_model(
    candidates_df: pd.DataFrame,
    references: List[str],
    model_configs: List[ModelConfig],
) -> pd.DataFrame:
    """Оценить top-1 от каждой модели отдельно (для сравнения)"""
    results = []
    
    for config in model_configs:
        col = f"{config.name}_0"  # top-1 кандидат
        if col in candidates_df.columns:
            preds = candidates_df[col].tolist()
            metrics = deep_past_score(preds, references)
            metrics["model"] = config.name
            results.append(metrics)
            print(f"  {config.name}: score={metrics['score']:.4f} "
                  f"(BLEU={metrics['bleu']:.2f}, chrF++={metrics['chrf++']:.2f})")
    
    return pd.DataFrame(results)

In [7]:
def translate(texts, model, tokenizer):
    model.to(device)
    batch_size = 16
    samples = 0
    all_outputs = []
    translations = []
    while(samples < len(texts)):
        batch = texts[samples:min(samples+batch_size, len(texts))]
        inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device) # Move inputs to the same device as the model
        outputs = model.generate(
        **inputs,
        # max_length=600,
        # max_new_tokens=600,
        max_new_tokens=512,
        num_beams=5,
        length_penalty=1.7,
        # top_p = 0.8,
        # temperature = 1.3
        )
        translations.append(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        samples += batch_size
    return translations

def remove_special_tokens(translations: str):
    pattern = r'</?[a-z|A-Z]+>'
    for index, translation in enumerate(translations):
        groups = re.findall(pattern, translation)
        for group in groups:
            # print(f"<<<<<<{groups}")
            if(not 'gap' in group):
                replacement = re.sub(pattern, "", group)
                translations[index] = translations[index].replace(group, replacement)
    return translations

determinatives_map = {"{d}":"{d}",
                  "{MUL}":"{MUL}",
                  "{mul}": "{MUL}",
                  "{KI}": "{ki}",
                  "{ki}": "{ki}",
                  "{LÚ}": "{LÚ}",
                  "{lú}": "{LÚ}",
                  "{lu2}": "{LÚ}",
                  "{lu₂}": "{LÚ}",
                  "{É}": "{É}",
                  "{é}": "{É}",
                  "{e₂}": "{É}",
                  "{URU}": "{URU}",
                  "{uru}": "{URU}",
                  "{KUR}":"{KUR}",
                  "{kur}": "{KUR}",
                  "{f}": "{f}",
                  "{mi}": "{f}",
                  "{mí}": "{f}",
                  "{munus}": "{f}",
                  "{1}": "{m}",
                  "{m}": "{m}",
                  "{GIŠ}": "{GIŠ}",
                  "{geš}": "{GIŠ}",
                  "{TÚG}":"{TÚG}",
                  "{túg}": "{TÚG}",
                  "{tug₂}": "{TÚG}",
                  "{tug}": "{TÚG}",
                  "{DUB}": "{DUB}",
                  "{dub}": "{DUB}",
                  "{ÍD}":"{ID}",
                  "{id}": "{ID}",
                  "{id₂}": "{ID}",
                  "{ID}": "{ID}",
                  "{MUŠEN}": "{MUŠEN}",
                  "{mušen}": "{MUŠEN}",
                  "{na₄}": "{na4}",
                  "{kuš}": "{KUŠ}",
                  "{KUŠ}": "{KUŠ}",
                  "{Ú}": "{Ú}",
                  "{u}": "{Ú}",
                  "{u₂}": "{Ú}"}

def normalize_determinatives(transliterations: str):
    texts_in_braces_pattern = r'\{[^\{\}]*\}'
    for index, transliteration in enumerate(transliterations):
        texts_in_braces = re.findall(texts_in_braces_pattern, transliteration)
        for braces_text in texts_in_braces:
          if(braces_text in determinatives_map):
            braces_text = braces_text.strip().replace(' ', '')
            transliteration = transliteration.replace(braces_text, determinatives_map[braces_text])
        transliterations[index] = transliteration
    return transliterations

def normalize_transliteration_chars_and_words(transliterations):
    for index, transliteration in enumerate(transliterations):
        for char, replacement in translit_character_map.items():
          transliteration = re.sub(char, replacement, transliteration)
        transliteration = re.sub(r'D[ÙUÚ]10', 'DÙG', transliteration)
        transliteration = re.sub(r'DÚG', 'DÙG', transliteration)
        transliteration = re.sub(r'š[AÀ]\.BA', 'ŠÀ.BA', transliteration, flags=re.IGNORECASE)
        transliteration = re.sub(r'GUŠKIN', 'KÙ.GI', transliteration)
        transliteration = re.sub(r'Pu-su-ke-en_6', 'Pu-šu-ke-en_6', transliteration)
        transliteration = re.sub(r'KÙ.KI', 'KÙ.GI', transliteration)
        transliteration = f"translate Old Assyrian transliteration to English: {transliteration}"
        transliterations[index] = transliteration
    return transliterations

def remove_all_scribal_characters(text: str):
    for scribal_character in scribal_characters:
      text = text.replace(scribal_character, "")
    return text

def normalize_sub_superscripts(texts: str):
    for index, text in enumerate(texts):
        for sub_sup, normal in subscript_to_normal_map.items():
            text = text.replace(sub_sup, normal)
            texts[index] = text
    return texts

def extract_grouped_text(text, patterns):
    all_groups = []
    for pattern in patterns:
      groups = re.findall(pattern, text)
      all_groups.extend(groups)
      # print(all_groups)
    return all_groups

def remove_scribal_notations(texts: list):
  # for scribal_notation in scribal_notations:
      for index, trans in enumerate(texts):
        groups = extract_grouped_text(texts[index], scribal_notation_patterns)
        if(groups):
          # print(len(groups))
          for i, group in enumerate(groups):
            # if(len(group)<7):
            #   print(group)
            # print(groups)
            groups[i] = remove_all_scribal_characters(groups[i])
            groups[i] = groups[i].replace(" . ", "")
            for key, value in determinatives_map.items():
              texts[index] = texts[index].replace(key, value)
                # print(group)
            if(len(groups[i]) == 2):
              groups[i] = ''
            groups[i] = groups[i].replace('[', '').replace(']', '').replace('<', '').replace('>', '').replace('˹', '').replace('˺', '')
            texts[index] = texts[index].replace(group, groups[i])
            # texts[index] = trans
      return texts

def standardize_text_breaks(texts: str):
      #text = re.sub(r'(?:\[?(?:\.{3,}|…)\]?\s*){2,}', '<big_gap>', text)
      gap_1 = r'(x\s?+){2,}'
      gap_2 = r'(…|\.{3,}\s?+){1,}'
      # gap_3 = r'(\s+x\s+)|(\[\s*x\s*\])|(x){2,}'
      collapsed_breaks = r'(<[a-z|_?]+>(\s+)?\(?\)?-?(\s+)?){2,}'
    
      trans_gaps_patterns = [r'(\s*x\s+){1,}', r'x{2,}']

      for index, text in enumerate(texts):
          text = re.sub(gap_2, '<big_gap>', text)
          text = re.sub(gap_1, '<big_gap>', text, flags=re.IGNORECASE)
          text = re.sub(r'\[?\s?x\s?\]?', "<gap>", text, flags=re.IGNORECASE)
          text = re.sub(collapsed_breaks, '<big_gap>', text)
          texts[index] = text
      return texts

def remove_residuals_from_breaks(texts: str):
    for index, text in enumerate(texts):
        text = text.replace('+', '')
        texts[index] = text
    return texts

def normalize(texts: str):
    for index, text in enumerate(texts):
        text = unicodedata.normalize('NFC', text)
        texts[index] = text
    return texts

def normalize_translation_fractions(texts: list):
    unicode_fraction_map = {'1/4': '¼',
                              '1/2': '½',
                              '1/3': '⅓',
                              '2/3': '⅔',
                              '5/6': '⅚',
                              '1/6': '⅙',
                               '3/4': '¾',
                               '1/5': '⅕',
                                '1/8': '⅛'}
    for index, text in enumerate(texts):
        for fraction, unicode_fraction in unicode_fraction_map.items():
            text = text.replace(fraction, unicode_fraction)
            text = text.replace('⁄', '/')
            texts[index] = text
    return texts

def normalize_transliteration_fractions(texts: list):
    unicode_fraction_map = {'¼': '1/4',
                              '½': '1/2',
                              '⅓': '1/3',
                              '⅔': '2/3',
                              '⅚': '5/6',
                              '⅙': '1/6',
                               '¾': '3/4',
                               '⅕': '1/5',
                                '⅛': '1/8'}
    for index, text in enumerate(texts):
        for unicode_fraction, fraction in unicode_fraction_map.items():
            text = text.replace(unicode_fraction, fraction)
            texts[index] = text
    return texts

In [8]:
# model = AutoModelForSeq2SeqLM.from_pretrained(model_path, local_files_only=True)
# tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

In [9]:
test_dataset = pd.read_csv(test_set_path)

In [10]:
# test_dataset

In [11]:
transliterations = test_dataset['transliteration'].tolist()
ids = test_dataset['id']

In [12]:
# if(len(transliterations)<1000):
#     a = 'ahsls'
#     b = a[1000]
# elif(len(transliterations)>=1000 and len(transliterations)<2000):
#     df = pd.DataFrame({'id': ids, 'translation': ids})
#     df.to_csv('/kaggle/working/submission.csv', index=False)

In [13]:
# # count = [transliteration for transliteration in transliterations if (len(transliteration.split(" ")) > 5 and len(transliteration.split(" "))<=20)]
# # # print(count)
# # if(len(count) < 1500):
# #     a = transliterations[0][1000000000000000000]
# # elif(len(count)>=1500 and len(count)<=3900):
# #     df = pd.DataFrame({'id': ids, 'translation': ids})
# #     df.to_csv('/kaggle/working/submission.csv', index=False)
# for transliteration in transliterations:
#     p = r'<gap>\s?+<big_gap>|<big_gap>\s?+<gap>'
#     q = re.search(p, transliteration)
#     if(q != None and q.group()):
#         print(p[100])

In [14]:
# transliterations = remove_scribal_notations(transliterations)
transliterations = normalize_sub_superscripts(transliterations)
# transliterations = remove_residuals_from_breaks(transliterations)
transliterations = normalize_transliteration_chars_and_words(transliterations)
transliterations = normalize_determinatives(transliterations)
transliterations = normalize_transliteration_fractions(transliterations)
# transliterations = standardize_text_breaks(transliterations)
# transliterations = normalize(transliterations)

In [15]:
# --- Определить модели ---
model_configs = [
    ModelConfig(
        name="mt5_large_drop",
        model_path="/kaggle/input/datasets/vitalykudelya/dpc26-models-2/aduah_MT_LARGE_from_zero_v1_continue_drop_checkpoint-1455/aduah_MT_LARGE_from_zero_v1_continue_drop_checkpoint-1455",
        num_beams=6,
        num_return=3,
        length_penalty=1.7,
        batch_size=1,
    ),
    ModelConfig(
        name="mt5_xl_bf16",
        model_path="/kaggle/input/datasets/vitalykudelya/dpc26-model-xl-runpod",
        num_beams=5,
        num_return=1,
        length_penalty=1.7,
        batch_size=1,
        torch_dtype=torch.bfloat16
    ),
    
    ModelConfig(
        name="mix_v2",
        model_path="/kaggle/input/datasets/vitalykudelya/dpc26-models-2/LARGE_MIXES_WITH_OLD_NEW_DATA_v2_53.93/LARGE_MIXES_WITH_OLD_NEW_DATA_v2",
        num_beams=6,
        num_return=3,
        length_penalty=1.7,
        batch_size=1,
    ),
    ModelConfig(
        name="drop_v1",
        model_path="/kaggle/input/datasets/vitalykudelya/dpc26-large-mixes-trained-with-test-drop-v1",
        num_beams=6,
        num_return=3,
        length_penalty=1.7,
        batch_size=1,
    ),
]

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Генерация кандидатов ---
candidates_df = generate_all_candidates(model_configs, transliterations, device)

# --- MBR выбор лучшего ---
predictions_df = select_best_candidates(candidates_df, strategy="chrf")

# --- Postprocessing (твой существующий код) ---
all_predictions = predictions_df["prediction"].tolist()
predictions_df


[mt5_large_drop] Loading model from /kaggle/input/datasets/vitalykudelya/dpc26-models-2/aduah_MT_LARGE_from_zero_v1_continue_drop_checkpoint-1455/aduah_MT_LARGE_from_zero_v1_continue_drop_checkpoint-1455
  beams=6, return=3, lp=1.7, batch=1


[mt5_large_drop] Generating: 100%|██████████| 4/4 [00:09<00:00,  2.34s/it]
`torch_dtype` is deprecated! Use `dtype` instead!


[mt5_large_drop] Done. Generated 4 × 3 candidates. VRAM free: 17.1 GB

[mt5_xl_bf16] Loading model from /kaggle/input/datasets/vitalykudelya/dpc26-model-xl-runpod
  beams=5, return=1, lp=1.7, batch=1


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[mt5_xl_bf16] Generating: 100%|██████████| 4/4 [00:56<00:00, 14.23s/it]


...

[mix_v2] Done. Generated 4 × 3 candidates. VRAM free: 17.1 GB

[drop_v1] Loading model from /kaggle/input/datasets/vitalykudelya/dpc26-large-mixes-trained-with-test-drop-v1
  beams=6, return=3, lp=1.7, batch=1


[drop_v1] Generating: 100%|██████████| 4/4 [00:14<00:00,  3.75s/it]


[drop_v1] Done. Generated 4 × 3 candidates. VRAM free: 17.1 GB

MBR Decoding (strategy=chrf)
  Candidates per example: 10
  Total examples: 4


MBR Decoding: 100%|██████████| 4/4 [00:00<00:00, 17.85it/s]


,text_idx,input_text,prediction
0,0,translate Old Assyrian transliteration to Engl...,"From the Kanesh colony to the dātu-payers, our..."
1,1,translate Old Assyrian transliteration to Engl...,In the tablet of the City it is recorded that ...
2,2,translate Old Assyrian transliteration to Engl...,"As soon as you hear our letter, whether he has..."
3,3,translate Old Assyrian transliteration to Engl...,I sent the copy of our tablet to every single ...


In [16]:
ids

0    0
1    1
2    2
3    3
Name: id, dtype: int64

In [17]:
# translations = translate(transliterations, model, tokenizer)
# all_predictions = []
# for translation in translations:
#     all_predictions.extend(translation)
translations = normalize_translation_fractions(all_predictions)

In [18]:
submission_file = pd.DataFrame({'id': ids, 'translation': translations})
submission_file.to_csv('/kaggle/working/submission.csv', index=False)

In [19]:
# translations
# transliterations = normalize(transliterations)

In [20]:
# def extract_unk_spans(text, tokenizer):
#     encoding = tokenizer(
#         text,
#         return_offsets_mapping=True,
#         add_special_tokens=False
#     )

#     unk_spans = []
#     for token_id, (start, end) in zip(encoding.input_ids, encoding.offset_mapping):
#         if token_id == tokenizer.unk_token_id:
#             unk_spans.append(text[start:end])

#     return unk_spans

In [21]:
# unks = []
# for t in transliterations:
#     unks.extend(extract_unk_spans(t, tokenizer))